# One-Sample t-Test Analysis: GNN (10-run distribution) vs. Random Forest (fixed value)
### Accuracy, MCC, precision / recall / F1, **AUROC and AUPRC**

**Input** (in the same folder as this notebook):

```
gcn_overall_multirun.csv      gcn_per_class_multirun.csv      gcn_auc_multirun.csv
gat_overall_multirun.csv      gat_per_class_multirun.csv      gat_auc_multirun.csv
gine_overall_multirun.csv     gine_per_class_multirun.csv     gine_auc_multirun.csv
gsage_overall_multirun.csv    gsage_per_class_multirun.csv    gsage_auc_multirun.csv
rf_auc.csv                    (Random Forest AUROC / AUPRC, one row per class + macro -- see section 2b)
```
The `*_overall_*` and `*_per_class_*` files come from `summarize_runs()`; the `*_auc_multirun.csv` files come from the
AUROC/AUPRC block of the multi-run cell. If the AUC files are absent the notebook still runs and simply skips the AUC section.

**Output**
- `FINAL_overall_ttest_vs_RF.csv` -- Accuracy & MCC, GNN vs RF, per architecture *(unchanged)*
- `FINAL_per_class_ttest_vs_RF.csv` -- Precision/Recall/F1 per class, per architecture *(unchanged)*
- `FINAL_f1_summary_all_architectures.csv` -- F1-only view *(unchanged)*
- `FINAL_table1_overall_meanstd.csv` -- Accuracy / MCC "mean +/- std" table *(unchanged)*
- **`FINAL_auc_ttest_vs_RF.csv`** -- macro and per-class AUROC / AUPRC, GNN vs RF, with t-tests *(new)*
- **`FINAL_table1_overall_with_auc_meanstd.csv`** -- Accuracy, MCC, macro AUROC, macro AUPRC *(new)*
- **`FINAL_table_auc_per_class_meanstd.csv`** -- per-class AUROC / AUPRC with RF and chance level *(new)*

The existing four files keep their exact format so downstream notebooks (figures, effect-size analysis) are unaffected.

**Method.** Random Forest is deterministic given the fixed train/test split, so it is treated as a fixed reference value
(mu0). Each GNN architecture was trained 10 times with different fixed seeds, giving n=10 independent outcomes per metric.
A one-sample t-test compares each GNN's 10-run distribution against RF's fixed value:

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \quad df = n - 1$$

The multirun CSVs store the *population* SD (NumPy default, ddof=0); it is converted to the *sample* SD (ddof=1) before
computing t and p.

**AUROC / AUPRC.** Both are computed one-vs-rest from the predicted class probabilities on the same test set (for the
GNNs, `exp` of the log-softmax output of the test nodes; for RF, `predict_proba`). The macro value is the mean of the four
per-class values. They depend only on how each model *ranks* molecules within a class, not on a decision threshold, so they
separate ranking quality from the operating point. Chance level is 0.5 for AUROC but equals the class prevalence for AUPRC
(for example ~0.05 for IA), so AUPRC values must be read against that baseline.

**Multiple comparisons.** Bonferroni (alpha = 0.05/4 = 0.0125) within each metric family across the 4 classes, per
architecture, for precision/recall/F1 and for per-class AUROC/AUPRC. The two macro AUC endpoints per architecture use
alpha = 0.05/2 = 0.025. (For Holm / Benjamini-Hochberg correction, effect sizes and confidence intervals, use the effect-size notebook.)

## 1. Setup

In [25]:
import os
import numpy as np
import pandas as pd
from scipy import stats

N_RUNS = 10
BONFERRONI_ALPHA = 0.05 / 4         # 4 classes tested per metric family, per architecture
BONFERRONI_ALPHA_MACRO = 0.05 / 2   # 2 macro AUC endpoints (AUROC, AUPRC) per architecture

INPUT_DIR = "."   # change if your CSVs live in a different folder
OUTPUT_DIR = "."  # change to write outputs elsewhere

AUC_FILE_SUFFIX = "_auc_multirun.csv"   # {prefix}_auc_multirun.csv, one per architecture
RF_AUC_CSV = "rf_auc.csv"               # Random Forest AUROC/AUPRC (preferred: full precision)

## 2. Random Forest fixed reference values

Taken directly from RF's `classification_report` / confusion matrix (identical across every notebook run, since RF is deterministic given the fixed split).

In [26]:
RF_OVERALL = {
    "Accuracy": 0.49,
    "MCC": 0.23,
}

RF_PER_CLASS = {
    "AA": {"precision": 0.25, "recall": 0.39, "f1": 0.30},
    "AI": {"precision": 0.07, "recall": 0.05, "f1": 0.06},
    "IA": {"precision": 0.49, "recall": 0.70, "f1": 0.58},
    "II": {"precision": 0.65, "recall": 0.41, "f1": 0.51},
}

CLASSES = ["AA", "AI", "IA", "II"]
METRICS_PER_CLASS = ["precision", "recall", "f1"]

# Map architecture name -> input file prefix used by your notebooks
ARCH_FILE_PREFIX = {
    "GCN": "gcn",
    "GAT": "gat",
    "GINE": "gine",
    "GraphSAGE": "gsage",
}

# Test-set size and class supports, used only for the AUPRC chance level (= class prevalence).
# Update if your split changes.
N_TEST = 523
SUPPORTS = {"AA": 64, "AI": 20, "IA": 167, "II": 272}

## 2b. Random Forest AUROC / AUPRC reference

Create `rf_auc.csv` **once, in the notebook where the Random Forest lives**, using the *same test set* as the GNNs, and copy it
next to this notebook (this avoids typing rounded numbers by hand):

```python
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize

proba = model_rf.predict_proba(data.x[test_idx])          # columns follow model_rf.classes_ = [0, 1, 2, 3]
y_true = np.asarray(data.y[test_idx])
y_bin = label_binarize(y_true, classes=[0, 1, 2, 3])

auroc_pc = roc_auc_score(y_true, proba, multi_class="ovr", average=None)
auprc_pc = np.array([average_precision_score(y_bin[:, c], proba[:, c]) for c in range(4)])

rf_auc = pd.DataFrame({"Class": ["macro", "AA", "AI", "IA", "II"],
                       "AUROC": [auroc_pc.mean(), *auroc_pc],
                       "AUPRC": [auprc_pc.mean(), *auprc_pc]})
rf_auc.to_csv("rf_auc.csv", index=False)
print(rf_auc.round(4))
```
`rf_auc.csv` must have the columns `Class` (macro, AA, AI, IA, II), `AUROC`, `AUPRC`. The macro AUROC equals your existing
`roc_auc_score(..., multi_class='ovr', average='macro')` call.

In [27]:
RF_AUC_FALLBACK = {   # used ONLY if rf_auc.csv is missing; fill in with full-precision values
    "macro": {"AUROC": None, "AUPRC": None},
    "AA":    {"AUROC": None, "AUPRC": None},
    "AI":    {"AUROC": None, "AUPRC": None},
    "IA":    {"AUROC": None, "AUPRC": None},
    "II":    {"AUROC": None, "AUPRC": None},
}

def load_rf_auc():
    """Returns {'macro'|class: {'AUROC': x, 'AUPRC': y}} for the fixed Random Forest reference."""
    path = f"{INPUT_DIR}/{RF_AUC_CSV}"
    if os.path.exists(path):
        df = pd.read_csv(path).set_index("Class")
        needed = ["macro"] + CLASSES
        missing = [c for c in needed if c not in df.index]
        if missing or not {"AUROC", "AUPRC"} <= set(df.columns):
            raise ValueError(f"{path} must have columns Class, AUROC, AUPRC and rows {needed}; missing rows: {missing}")
        return {c: {"AUROC": float(df.loc[c, "AUROC"]), "AUPRC": float(df.loc[c, "AUPRC"])} for c in needed}
    if all(v is not None for d in RF_AUC_FALLBACK.values() for v in d.values()):
        return RF_AUC_FALLBACK
    raise FileNotFoundError(f"{path} not found and RF_AUC_FALLBACK is empty -- create rf_auc.csv as shown in section 2b.")

AUC_ARCH_FILES = {arch: f"{INPUT_DIR}/{prefix}{AUC_FILE_SUFFIX}" for arch, prefix in ARCH_FILE_PREFIX.items()}
AUC_AVAILABLE_ARCHS = [a for a, f in AUC_ARCH_FILES.items() if os.path.exists(f)]
if AUC_AVAILABLE_ARCHS:
    RF_AUC = load_rf_auc()
    skipped = [a for a in ARCH_FILE_PREFIX if a not in AUC_AVAILABLE_ARCHS]
    print("AUC analysis will use:", AUC_AVAILABLE_ARCHS, "| missing (skipped):", skipped)
else:
    RF_AUC = None
    print("No *_auc_multirun.csv files found -- the AUROC/AUPRC section will be skipped.")

AUC analysis will use: ['GCN', 'GAT', 'GINE', 'GraphSAGE'] | missing (skipped): []


## 3. One-sample t-test helper

In [28]:
def one_sample_ttest_vs_fixed(mu0, xbar, s_pop, n=N_RUNS):
    """
    One-sample t-test of a GNN's n-run distribution (mean=xbar, population
    std=s_pop) against a fixed reference value mu0 (Random Forest).
    Returns (t_statistic, p_value). Returns (nan, nan) if the GNN showed zero
    variance across runs (t-test undefined).
    """
    if s_pop == 0 or np.isnan(s_pop):
        return np.nan, np.nan
    s_sample = s_pop * np.sqrt(n / (n - 1))  # population std (ddof=0) -> sample std (ddof=1)
    se = s_sample / np.sqrt(n)
    t = (xbar - mu0) / se
    df = n - 1
    p = 2 * (1 - stats.t.cdf(abs(t), df))
    return t, p

## 4. Load each architecture's multi-run summary CSVs and run the t-tests

In [29]:
def analyze():
    overall_rows = []
    per_class_rows = []

    for arch, prefix in ARCH_FILE_PREFIX.items():
        overall_df = pd.read_csv(f"{INPUT_DIR}/{prefix}_overall_multirun.csv").set_index("Metric")
        per_class_df = pd.read_csv(f"{INPUT_DIR}/{prefix}_per_class_multirun.csv").set_index("Class")

        # ---- overall: Accuracy, MCC ----
        for metric in ["Accuracy", "MCC"]:
            xbar = overall_df.loc[metric, "Mean"]
            s = overall_df.loc[metric, "Std"]
            mu0 = RF_OVERALL[metric]
            t, p = one_sample_ttest_vs_fixed(mu0, xbar, s)
            overall_rows.append({
                "Architecture": arch, "Metric": metric,
                "RF": mu0, "GNN_mean": xbar, "GNN_std": s,
                "t_stat": t, "p_value": p,
            })

        # ---- per-class: precision, recall, f1 ----
        for cls in CLASSES:
            for metric, col in [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]:
                xbar = per_class_df.loc[cls, f"{col}_mean"]
                s = per_class_df.loc[cls, f"{col}_std"]
                mu0 = RF_PER_CLASS[cls][metric]
                t, p = one_sample_ttest_vs_fixed(mu0, xbar, s)
                sig = (p < BONFERRONI_ALPHA) if not np.isnan(p) else False
                per_class_rows.append({
                    "Architecture": arch, "Class": cls, "Metric": metric,
                    "RF": mu0, "GNN_mean": xbar, "GNN_std": s,
                    "diff": xbar - mu0, "t_stat": t, "p_value": p,
                    "significant_bonf": sig,
                })

    overall_result = pd.DataFrame(overall_rows)
    per_class_result = pd.DataFrame(per_class_rows)
    return overall_result, per_class_result


overall_result, per_class_result = analyze()
#overall_result.head()
overall_result

,Architecture,Metric,RF,GNN_mean,GNN_std,t_stat,p_value
0,GCN,Accuracy,0.49,0.523901,0.022001,4.622595,0.001250
1,GCN,MCC,0.23,0.218763,0.030760,-1.095890,0.301584
2,GAT,Accuracy,0.49,0.482983,0.026785,-0.785948,0.452093
3,GAT,MCC,0.23,0.241178,0.028565,1.173950,0.270543
4,GINE,Accuracy,0.49,0.449522,0.056581,-2.146184,0.060411
5,GINE,MCC,0.23,0.208893,0.044212,-1.432194,0.185890
6,GraphSAGE,Accuracy,0.49,0.521033,0.022154,4.202258,0.002299
7,GraphSAGE,MCC,0.23,0.235014,0.021845,0.688645,0.508408


## 4b. AUROC and AUPRC: one-sample t-tests against the fixed RF values

Same test as above, applied to macro and per-class AUROC and AUPRC.

In [30]:
def analyze_auc():
    if not AUC_AVAILABLE_ARCHS:
        return pd.DataFrame()
    rows = []
    for arch in AUC_AVAILABLE_ARCHS:
        auc_df = pd.read_csv(AUC_ARCH_FILES[arch]).set_index("Class")
        for cls in ["macro"] + CLASSES:
            for metric in ["AUROC", "AUPRC"]:
                xbar = auc_df.loc[cls, f"{metric}_mean"]
                s = auc_df.loc[cls, f"{metric}_std"]
                mu0 = RF_AUC[cls][metric]
                t, p = one_sample_ttest_vs_fixed(mu0, xbar, s)
                alpha = BONFERRONI_ALPHA_MACRO if cls == "macro" else BONFERRONI_ALPHA
                rows.append({
                    "Architecture": arch,
                    "Scope": "overall" if cls == "macro" else "per-class",
                    "Class": cls, "Metric": metric,
                    "RF": mu0, "GNN_mean": xbar, "GNN_std": s,
                    "diff": xbar - mu0, "t_stat": t, "p_value": p,
                    "significant_bonf": (p < alpha) if not np.isnan(p) else False,
                })
    return pd.DataFrame(rows)


auc_result = analyze_auc()
#auc_result.head()
auc_result

,Architecture,Scope,Class,Metric,RF,GNN_mean,GNN_std,diff,t_stat,p_value,significant_bonf
0,GCN,overall,macro,AUROC,0.634806,0.664504,0.016999,0.029698,5.241010,5.340430e-04,True
1,GCN,overall,macro,AUPRC,0.346557,0.378543,0.009733,0.031987,9.859690,4.024340e-06,True
2,GCN,per-class,AA,AUROC,0.596609,0.608984,0.034287,0.012374,1.082686,3.071007e-01,False
3,GCN,per-class,AA,AUPRC,0.209105,0.185167,0.020825,-0.023938,-3.448343,7.293772e-03,True
4,GCN,per-class,AI,AUROC,0.680815,0.680219,0.051341,-0.000596,-0.034851,9.729594e-01,False
5,GCN,per-class,AI,AUPRC,0.082911,0.082981,0.033667,0.000070,0.006210,9.951804e-01,False
6,GCN,per-class,IA,AUROC,0.734408,0.708805,0.015159,-0.025602,-5.066669,6.749102e-04,True
7,GCN,per-class,IA,AUPRC,0.533386,0.564597,0.028577,0.031211,3.276499,9.582699e-03,True
8,GCN,per-class,II,AUROC,0.527390,0.660007,0.019016,0.132617,20.921736,6.099953e-09,True
9,GCN,per-class,II,AUPRC,0.560826,0.681430,0.020218,0.120604,17.895522,2.415022e-08,True


## 5. Derived tables (F1-only summary, human-readable Table 1)

In [31]:
def build_f1_summary(per_class_result):
    f1_summary = per_class_result[per_class_result["Metric"] == "f1"].copy()
    f1_summary["direction"] = f1_summary["diff"].apply(lambda d: "GNN higher" if d > 0 else "RF higher")
    return f1_summary


def build_table1(overall_result):
    rows = [{"Model": "Random Forest",
             "Accuracy": f"{RF_OVERALL['Accuracy']:.3f} (fixed)",
             "MCC": f"{RF_OVERALL['MCC']:.3f} (fixed)"}]
    for arch in ARCH_FILE_PREFIX:
        sub = overall_result[overall_result.Architecture == arch].set_index("Metric")
        rows.append({
            "Model": arch,
            "Accuracy": f"{sub.loc['Accuracy','GNN_mean']:.3f} +/- {sub.loc['Accuracy','GNN_std']:.3f}",
            "MCC": f"{sub.loc['MCC','GNN_mean']:.3f} +/- {sub.loc['MCC','GNN_std']:.3f}",
        })
    return pd.DataFrame(rows)


f1_summary = build_f1_summary(per_class_result)
table1 = build_table1(overall_result)

In [32]:
def build_table1_with_auc(overall_result, auc_result):
    """Accuracy, MCC, macro AUROC, macro AUPRC: mean +/- std (RF fixed)."""
    rows = [{"Model": "Random Forest",
             "Accuracy": f"{RF_OVERALL['Accuracy']:.3f} (fixed)", "MCC": f"{RF_OVERALL['MCC']:.3f} (fixed)",
             "Macro AUROC": f"{RF_AUC['macro']['AUROC']:.3f} (fixed)", "Macro AUPRC": f"{RF_AUC['macro']['AUPRC']:.3f} (fixed)"}]
    for arch in AUC_AVAILABLE_ARCHS:
        o = overall_result[overall_result.Architecture == arch].set_index("Metric")
        a = auc_result[(auc_result.Architecture == arch) & (auc_result.Class == "macro")].set_index("Metric")
        rows.append({"Model": arch,
            "Accuracy": f"{o.loc['Accuracy','GNN_mean']:.3f} +/- {o.loc['Accuracy','GNN_std']:.3f}",
            "MCC": f"{o.loc['MCC','GNN_mean']:.3f} +/- {o.loc['MCC','GNN_std']:.3f}",
            "Macro AUROC": f"{a.loc['AUROC','GNN_mean']:.3f} +/- {a.loc['AUROC','GNN_std']:.3f}",
            "Macro AUPRC": f"{a.loc['AUPRC','GNN_mean']:.3f} +/- {a.loc['AUPRC','GNN_std']:.3f}"})
    return pd.DataFrame(rows)


def build_auc_per_class_table(auc_result):
    """Per-class AUROC / AUPRC: mean +/- std, with RF and the chance level (AUROC 0.5, AUPRC = prevalence)."""
    cols = [f"{m} {c}" for m in ["AUROC", "AUPRC"] for c in CLASSES]
    rows = [{"Model": "Random Forest", **{f"{m} {c}": f"{RF_AUC[c][m]:.3f} (fixed)" for m in ["AUROC", "AUPRC"] for c in CLASSES}}]
    for arch in AUC_AVAILABLE_ARCHS:
        sub = auc_result[auc_result.Architecture == arch].set_index(["Class", "Metric"])
        rows.append({"Model": arch, **{f"{m} {c}": f"{sub.loc[(c, m), 'GNN_mean']:.3f} +/- {sub.loc[(c, m), 'GNN_std']:.3f}"
                                       for m in ["AUROC", "AUPRC"] for c in CLASSES}})
    rows.append({"Model": "Chance level", **{f"AUROC {c}": "0.500" for c in CLASSES},
                 **{f"AUPRC {c}": f"{SUPPORTS[c] / N_TEST:.3f}" for c in CLASSES}})
    return pd.DataFrame(rows)[["Model"] + cols]


if AUC_AVAILABLE_ARCHS:
    table1_auc = build_table1_with_auc(overall_result, auc_result)
    auc_per_class_table = build_auc_per_class_table(auc_result)
else:
    table1_auc = auc_per_class_table = None

## 6. Results

In [33]:
print("=== Table 1: overall performance (mean +/- std) ===")
print(table1.to_string(index=False))

=== Table 1: overall performance (mean +/- std) ===
        Model        Accuracy             MCC
Random Forest   0.490 (fixed)   0.230 (fixed)
          GCN 0.524 +/- 0.022 0.219 +/- 0.031
          GAT 0.483 +/- 0.027 0.241 +/- 0.029
         GINE 0.450 +/- 0.057 0.209 +/- 0.044
    GraphSAGE 0.521 +/- 0.022 0.235 +/- 0.022


In [34]:
print("=== Significant per-class results (Bonferroni-corrected, alpha=0.0125) ===")
sig = per_class_result[per_class_result.significant_bonf]
sig[["Architecture", "Class", "Metric", "RF", "GNN_mean", "diff", "p_value"]]

=== Significant per-class results (Bonferroni-corrected, alpha=0.0125) ===


,Architecture,Class,Metric,RF,GNN_mean,diff,p_value
1,GCN,AA,recall,0.39,0.189062,-0.200938,2.075477e-08
2,GCN,AA,f1,0.30,0.208381,-0.091619,3.402804e-05
7,GCN,IA,recall,0.70,0.581437,-0.118563,1.673724e-03
8,GCN,IA,f1,0.58,0.527100,-0.052900,1.608590e-03
10,GCN,II,recall,0.41,0.604044,0.194044,1.922940e-05
11,GCN,II,f1,0.51,0.617195,0.107195,5.012942e-06
12,GAT,AA,precision,0.25,0.225951,-0.024049,9.691191e-03
13,GAT,AA,recall,0.39,0.273438,-0.116563,1.799135e-06
14,GAT,AA,f1,0.30,0.245459,-0.054541,1.672077e-06
16,GAT,AI,recall,0.05,0.240000,0.190000,2.707454e-03


In [35]:
if AUC_AVAILABLE_ARCHS:
    print("=== Table 1 with AUROC / AUPRC (mean +/- std; RF fixed) ===")
    print(table1_auc.to_string(index=False))
    print("\n=== Per-class AUROC / AUPRC (mean +/- std; RF fixed; chance level in the last row) ===")
    print(auc_per_class_table.to_string(index=False))
    print("\n=== Significant AUROC / AUPRC differences vs RF (per-class: alpha=0.0125; macro: alpha=0.025) ===")
    display_cols = ["Architecture", "Class", "Metric", "RF", "GNN_mean", "diff", "p_value"]
    sig_auc = auc_result[auc_result.significant_bonf][display_cols]
    print(sig_auc.round(4).to_string(index=False) if len(sig_auc) else "  none")
else:
    print("AUROC/AUPRC section skipped (no *_auc_multirun.csv files).")

=== Table 1 with AUROC / AUPRC (mean +/- std; RF fixed) ===
        Model        Accuracy             MCC     Macro AUROC     Macro AUPRC
Random Forest   0.490 (fixed)   0.230 (fixed)   0.635 (fixed)   0.347 (fixed)
          GCN 0.524 +/- 0.022 0.219 +/- 0.031 0.665 +/- 0.017 0.379 +/- 0.010
          GAT 0.483 +/- 0.027 0.241 +/- 0.029 0.700 +/- 0.018 0.407 +/- 0.018
         GINE 0.450 +/- 0.057 0.209 +/- 0.044 0.677 +/- 0.028 0.383 +/- 0.023
    GraphSAGE 0.521 +/- 0.022 0.235 +/- 0.022 0.650 +/- 0.011 0.375 +/- 0.006

=== Per-class AUROC / AUPRC (mean +/- std; RF fixed; chance level in the last row) ===
        Model        AUROC AA        AUROC AI        AUROC IA        AUROC II        AUPRC AA        AUPRC AI        AUPRC IA        AUPRC II
Random Forest   0.597 (fixed)   0.681 (fixed)   0.734 (fixed)   0.527 (fixed)   0.209 (fixed)   0.083 (fixed)   0.533 (fixed)   0.561 (fixed)
          GCN 0.609 +/- 0.034 0.680 +/- 0.051 0.709 +/- 0.015 0.660 +/- 0.019 0.185 +/- 0.021 0.083 

## 7. Save outputs

In [36]:
overall_result.to_csv(f"{OUTPUT_DIR}/FINAL_overall_ttest_vs_RF.csv", index=False)
per_class_result.to_csv(f"{OUTPUT_DIR}/FINAL_per_class_ttest_vs_RF.csv", index=False)
f1_summary.to_csv(f"{OUTPUT_DIR}/FINAL_f1_summary_all_architectures.csv", index=False)
table1.to_csv(f"{OUTPUT_DIR}/FINAL_table1_overall_meanstd.csv", index=False)

print("Saved: FINAL_overall_ttest_vs_RF.csv, FINAL_per_class_ttest_vs_RF.csv, "
      "FINAL_f1_summary_all_architectures.csv, FINAL_table1_overall_meanstd.csv")

if AUC_AVAILABLE_ARCHS:
    auc_result.to_csv(f"{OUTPUT_DIR}/FINAL_auc_ttest_vs_RF.csv", index=False)
    table1_auc.to_csv(f"{OUTPUT_DIR}/FINAL_table1_overall_with_auc_meanstd.csv", index=False)
    auc_per_class_table.to_csv(f"{OUTPUT_DIR}/FINAL_table_auc_per_class_meanstd.csv", index=False)
    print("Saved: FINAL_auc_ttest_vs_RF.csv, FINAL_table1_overall_with_auc_meanstd.csv, FINAL_table_auc_per_class_meanstd.csv")

Saved: FINAL_overall_ttest_vs_RF.csv, FINAL_per_class_ttest_vs_RF.csv, FINAL_f1_summary_all_architectures.csv, FINAL_table1_overall_meanstd.csv
Saved: FINAL_auc_ttest_vs_RF.csv, FINAL_table1_overall_with_auc_meanstd.csv, FINAL_table_auc_per_class_meanstd.csv
